In [ ]:
!pip install -q bnlp_toolkit indic-nlp-library

In [ ]:
import os
import re
import random
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
from bnlp import SentencepieceTokenizer, SentencepieceTrainer, NLTKTokenizer, BasicTokenizer
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

In [ ]:
# Configuration
POEM_DIR = Path("bangla_kobita")
MAX_POEMS = 2000

# Subword N-gram configuration
N = 4
SP_VOCAB_SIZE = 4000         # SentencePiece BPE vocabulary size
SP_MODEL_PREFIX = "bn_sp_poetry"
OVERRIDE_REGEN = False       # Set to True to force retraining/regenerating model even if .model and .vocab exist

# Generation settings
GENERATION_MODE = "both"     # Options: "mle", "seeded", "both"
RANDOM_SEED = 42
TEMPERATURE = 0.3
TOP_K_SAMPLING = None       # Set to None to keep all vocabulary choices without top-k filtering
MIN_SUBWORDS_PER_LINE = 6    # Ensures lines have sufficient subword tokens before line breaks
MAX_GENERATED_SUBWORDS = 200
MAX_CONSECUTIVE_NEWLINES = 2

# Repetition control
NO_REPEAT_NGRAM = 4
MAX_SAME_LINE_REPEAT = 2

START_PROMPT = "তারেক রহমান"

START = "<s>"
END = "</s>"
NEWLINE = "<nl>"
UNK = "<unk>"

WHITESPACE_RE = re.compile(r"\s+")
normalizer = IndicNormalizerFactory().get_normalizer("bn")

In [27]:
def load_and_normalize_poems(folder, limit=None):
    """Load .txt poems and apply NFC Unicode & Indic normalization."""
    folder = Path(folder)
    if not folder.exists(): raise FileNotFoundError(f"Poem folder not found: {folder.resolve()}")
    files = sorted(folder.glob("*.txt"))
    if not files: raise FileNotFoundError(f"No .txt files found in: {folder.resolve()}")
    if limit is not None: files = files[:limit]

    poems = []
    for file in files:
        try: text = file.read_text(encoding="utf-8")
        except UnicodeDecodeError: text = file.read_text(encoding="utf-8", errors="ignore")
        text = text.strip()
        if text:
            text = normalizer.normalize(unicodedata.normalize("NFC", text))
            poems.append(text)
    return poems

poems = load_and_normalize_poems(POEM_DIR, MAX_POEMS)
print("Poems loaded:", len(poems))

Poems loaded: 2000


In [28]:
def get_bnlp_tokenizer(poems, model_prefix, vocab_size, override=False):
    """Train or load a BNLP SentencepieceTokenizer on the raw poem corpus."""
    model_file = f"{model_prefix}.model"
    vocab_file = f"{model_prefix}.vocab"

    if not override and os.path.exists(model_file) and os.path.exists(vocab_file):
        print(f"Loading existing BNLP Sentencepiece model from {model_file} and {vocab_file}...")
        return SentencepieceTokenizer(model_file)

    print(f"Training BNLP Sentencepiece model ({model_prefix})...")
    temp_corpus_path = "temp_sp_corpus.txt"

    with open(temp_corpus_path, "w", encoding="utf-8") as f:
        for poem in poems:
            f.write(poem + "\n\n")

    trainer = SentencepieceTrainer(temp_corpus_path, vocab_size, model_prefix)
    trainer.train()

    if os.path.exists(temp_corpus_path):
        os.remove(temp_corpus_path)

    return SentencepieceTokenizer(model_file)

sp = get_bnlp_tokenizer(poems, SP_MODEL_PREFIX, SP_VOCAB_SIZE, override=OVERRIDE_REGEN)
print("BNLP SentencePiece Vocabulary Size:", sp.model.get_piece_size())

I0000 00:00:1786236485.998815   19805 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: temp_sp_corpus.txt
  input_format: 
  model_prefix: bn_sp_poetry
  model_type: BPE
  vocab_size: 4000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <nl>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <p

SentencePiece Vocabulary Size: 4000


I0000 00:00:1786236486.458741   19805 bpe_model_trainer.cc:321] Added: freq=757 size=200 all=12341 active=12262 piece=▁দেখ
I0000 00:00:1786236486.462535   19805 bpe_model_trainer.cc:321] Added: freq=644 size=220 all=13023 active=12944 piece=ুলে
I0000 00:00:1786236486.465595   19805 bpe_model_trainer.cc:321] Added: freq=580 size=240 all=13586 active=13507 piece=ান্ত
I0000 00:00:1786236486.467795   19805 bpe_model_trainer.cc:321] Added: freq=532 size=260 all=14069 active=13990 piece=▁কত
I0000 00:00:1786236486.470458   19805 bpe_model_trainer.cc:321] Added: freq=494 size=280 all=14870 active=14791 piece=▁নব
I0000 00:00:1786236486.472980   19805 bpe_model_trainer.cc:321] Added: freq=468 size=300 all=15463 active=15384 piece=▁জল
I0000 00:00:1786236486.475816   19805 bpe_model_trainer.cc:321] Added: freq=433 size=320 all=15987 active=15908 piece=শা
I0000 00:00:1786236486.478472   19805 bpe_model_trainer.cc:321] Added: freq=404 size=340 all=16695 active=16616 piece=নু
I0000 00:00:1786236486.4

In [29]:
def prepare_subword_corpus(poems, sp, n):
    """Tokenize poems into subword sequence vectors using BNLP tokenizer."""
    sequences = []
    for poem in poems:
        lines = poem.splitlines()
        seq = [] if n <= 1 else [START] * (n - 1)
        added_content = False

        for raw_line in lines:
            line = WHITESPACE_RE.sub(" ", raw_line).strip()
            if line:
                subwords = sp.tokenize(line)
                seq.extend(subwords)
                seq.append(NEWLINE)
                added_content = True
            elif added_content:
                seq.append(NEWLINE)

        if added_content:
            while seq and seq[-1] == NEWLINE: seq.pop()
            seq.append(END)
            sequences.append(seq)
    return sequences

sequences = prepare_subword_corpus(poems, sp, N)
total_subwords = sum(len(seq) for seq in sequences)

print("Subword Sequences:", len(sequences))
print("Total Subword Tokens:", total_subwords)

Subword Sequences: 2000
Total Subword Tokens: 496155


In [30]:
def format_generated_subwords(output_subwords, sp):
    """Decode generated subwords back into clean Bengali text using BNLP tokenizer."""
    if not output_subwords: return ""

    lines = []
    current_line_tokens = []

    for token in output_subwords:
        if token == NEWLINE:
            if current_line_tokens:
                lines.append(sp.model.decode_pieces(current_line_tokens))
                current_line_tokens = []
            else:
                lines.append("")
        else:
            current_line_tokens.append(token)

    if current_line_tokens:
        lines.append(sp.model.decode_pieces(current_line_tokens))

    cleaned, blank_count = [], 0
    for line in lines:
        line = line.strip()
        if line == "":
            blank_count += 1
            if blank_count <= 1: cleaned.append(line)
        else:
            blank_count = 0
            cleaned.append(line)

    while cleaned and cleaned[0] == "": cleaned.pop(0)
    while cleaned and cleaned[-1] == "": cleaned.pop()
    return "\n".join(cleaned)

In [31]:
def creates_repeated_ngram(tokens, candidate, n):
    """Return True if adding candidate creates an n-gram already in tokens."""
    if not n or n <= 0 or candidate == END or len(tokens) + 1 < n: return False
    target = tuple(tokens[-(n - 1):]) + (candidate,)
    return any(tuple(tokens[i:i + n]) == target for i in range(len(tokens) - n + 1))

In [ ]:
class NGramModel:
    """Interpolated Kneser-Ney n-gram model operating on subwords."""

    def __init__(self, n=4, unk_count=1.0, discount=None):
        if n < 1: raise ValueError("n must be >= 1")
        self.n = n
        self.unk_token = UNK
        self.unk_count = unk_count
        self.fixed_discount = discount
        self.vocab = {UNK, END, NEWLINE}
        self.raw_counts = {order: defaultdict(Counter) for order in range(1, n + 1)}
        self.continuation_counts = {order: defaultdict(Counter) for order in range(1, n)}
        self.discounts = {order: 0.0 for order in range(1, n + 1)}
        self._dist_cache = {}

    def train(self, sequences):
        """Build n-gram counts and smoothed distributions."""
        if sequences and isinstance(sequences[0], str): sequences = [sequences]
        for seq in sequences:
            if not seq: continue
            for word in seq:
                if word != START: self.vocab.add(word)
            for i, word in enumerate(seq):
                if word == START: continue
                for order in range(1, self.n + 1):
                    if i >= order - 1:
                        history = tuple(seq[i - order + 1:i])
                        self.raw_counts[order][history][word] += 1

        self._build_continuation_counts()
        self._add_unknown_fallback()
        self._build_discounts()
        self._dist_cache.clear()

    def _build_continuation_counts(self):
        self.continuation_counts = {order: defaultdict(Counter) for order in range(1, self.n)}
        for order in range(1, self.n):
            seen = defaultdict(set)
            for history, word_counts in self.raw_counts[order + 1].items():
                if not history: continue
                left_context, lower_history = history[0], history[1:]
                for word in word_counts:
                    seen[(lower_history, word)].add(left_context)
            for (lower_history, word), lefts in seen.items():
                self.continuation_counts[order][lower_history][word] = len(lefts)

    def _add_unknown_fallback(self):
        self.vocab.add(UNK)
        if self.unk_count <= 0: return
        if self.n == 1: self.raw_counts[1][()][UNK] += self.unk_count
        else: self.continuation_counts[1][()][UNK] += self.unk_count

    def _build_discounts(self):
        for order in range(1, self.n + 1):
            counts = self.raw_counts[order] if order == self.n else self.continuation_counts[order]
            self.discounts[order] = self._compute_discount(counts)

    def _compute_discount(self, counts_by_history):
        if self.fixed_discount is not None: return float(self.fixed_discount)
        n1 = sum(1 for counter in counts_by_history.values() for c in counter.values() if c == 1)
        n2 = sum(1 for counter in counts_by_history.values() for c in counter.values() if c == 2)
        if n1 == 0 or n2 == 0: return 0.75
        return max(0.05, min(0.95, n1 / (n1 + 2.0 * n2)))

    def _normalize_token(self, token):
        return token if token == START or token in self.vocab else UNK

    def _canonical_history(self, history):
        if self.n == 1: return ()
        if history is None: history = []
        tokens = list(history)[-(self.n - 1):]
        return tuple(self._normalize_token(t) for t in tokens)

    def _prob_dist(self, history):
        history = () if self.n == 1 else tuple(history[-(self.n - 1):])
        if history in self._dist_cache: return self._dist_cache[history]

        order = len(history) + 1
        if order == 1:
            counter = self.raw_counts[1][()] if self.n == 1 else self.continuation_counts[1].get((), {})
            total = sum(counter.values())
            dist = {UNK: 1.0} if total <= 0 else {w: c / total for w, c in counter.items()}
            self._dist_cache[history] = dist
            return dist

        counts = self.raw_counts[order] if order == self.n else self.continuation_counts[order]
        counter = counts.get(history, {})
        total = sum(counter.values())

        if total <= 0:
            dist = self._prob_dist(history[1:])
            self._dist_cache[history] = dist
            return dist

        discount = self.discounts[order]
        lower = self._prob_dist(history[1:])
        dist = {w: max(c - discount, 0.0) / total for w, c in counter.items() if c > discount}
        backoff_weight = discount * len(counter) / total

        if backoff_weight > 0:
            for w, p in lower.items():
                dist[w] = dist.get(w, 0.0) + backoff_weight * p

        total_prob = sum(dist.values())
        dist = {w: p / total_prob for w, p in dist.items()} if total_prob > 0 else lower
        self._dist_cache[history] = dist
        return dist

    def get_choices(self, history, require_word=None):
        dist = dict(self._prob_dist(self._canonical_history(history)))
        dist.pop(START, None)
        if require_word is None: return dist
        required = self._normalize_token(require_word)
        return dist if required in dist else {}

    def generate(
        self,
        sp_model,
        max_length=200,
        temperature=0.7,
        top_k=TOP_K_SAMPLING,
        min_subwords_per_line=MIN_SUBWORDS_PER_LINE,
        seed=None,
        max_consecutive_newlines=MAX_CONSECUTIVE_NEWLINES,
        prompt=None,
        include_prompt=True,
        mode="mle",
        no_repeat_ngram=NO_REPEAT_NGRAM,
        max_same_line_repeat=MAX_SAME_LINE_REPEAT,
    ):
        mode = str(mode).lower().strip()
        if mode not in {"mle", "seeded"}:
            raise ValueError("generation mode must be either \"mle\" or \"seeded\"")

        rng = random.Random(seed) if seed is not None else random.Random()

        raw_prompt_subwords = sp_model.tokenize(prompt) if prompt else []
        raw_prompt_subwords = [t for t in raw_prompt_subwords if t not in (START, END)]
        model_prompt_subwords = [self._normalize_token(t) for t in raw_prompt_subwords]

        if raw_prompt_subwords:
            context = ([] if self.n == 1 else [START] * (self.n - 1)) + model_prompt_subwords
            history = context[-(self.n - 1):] if self.n > 1 else []
            output_tokens = list(raw_prompt_subwords) if include_prompt else []
        else:
            history = [] if self.n == 1 else [START] * (self.n - 1)
            output_tokens = []

        consecutive_newlines = sum(1 for t in reversed(model_prompt_subwords) if t == NEWLINE)

        current_line = []
        for token in reversed(output_tokens):
            if token == NEWLINE: break
            current_line.insert(0, token)

        last_line, same_line_count = None, 0

        for _ in range(max_length):
            choices = self.get_choices(history)
            choices.pop(START, None)
            choices.pop(UNK, None)

            # Prevent line breaks before reaching minimum subwords threshold
            if len(current_line) < min_subwords_per_line:
                choices.pop(NEWLINE, None)

            if not output_tokens or consecutive_newlines >= max_consecutive_newlines:
                choices.pop(NEWLINE, None)

            if not choices:
                choices = {END: 1.0}

            if no_repeat_ngram and no_repeat_ngram > 0:
                filtered = {w: p for w, p in choices.items() if w == END or not creates_repeated_ngram(output_tokens, w, no_repeat_ngram)}
                choices = filtered if filtered else ({END: choices[END]} if END in choices else {END: 1.0})

            # Top-K filtering (only applies if top_k is set to an integer > 0)
            if mode != "mle" and top_k is not None and top_k > 0 and len(choices) > top_k:
                top_items = sorted(choices.items(), key=lambda x: x[1], reverse=True)[:top_k]
                choices = dict(top_items)

            words, probs = list(choices.keys()), list(choices.values())

            if mode == "mle" or temperature is None or temperature <= 0:
                word = min(choices.items(), key=lambda x: (-x[1], x[0]))[0]
            elif temperature != 1.0:
                weighted = [p ** (1.0 / temperature) for p in probs]
                total = sum(weighted)
                word = rng.choice(words) if total <= 0 else rng.choices(words, weights=[w / total for w in weighted], k=1)[0]
            else:
                word = rng.choices(words, weights=probs, k=1)[0]

            if word == END: break

            output_tokens.append(word)

            if word == NEWLINE:
                consecutive_newlines += 1
                line_tuple = tuple(current_line)
                if line_tuple == last_line:
                    same_line_count += 1
                else:
                    same_line_count = 1
                    last_line = line_tuple
                current_line = []
                if max_same_line_repeat and same_line_count >= max_same_line_repeat: break
            else:
                consecutive_newlines = 0
                current_line.append(word)

            if self.n > 1:
                history.append(word)
                history = history[-(self.n - 1):]

        return format_generated_subwords(output_tokens, sp_model)

In [33]:
model = NGramModel(N)
model.train(sequences)

print("Trained Subword N-Gram Vocabulary Size:", len(model.vocab))

Trained Subword N-Gram Vocabulary Size: 4059


In [34]:
def generate_with_mode(mode):
    mode = mode.lower().strip()
    temperature = None if mode == "mle" else TEMPERATURE
    return model.generate(
        sp_model=sp,
        max_length=MAX_GENERATED_SUBWORDS,
        temperature=temperature,
        top_k=TOP_K_SAMPLING,
        min_subwords_per_line=MIN_SUBWORDS_PER_LINE,
        seed=RANDOM_SEED,
        max_consecutive_newlines=MAX_CONSECUTIVE_NEWLINES,
        prompt=START_PROMPT,
        include_prompt=True,
        mode=mode,
        no_repeat_ngram=NO_REPEAT_NGRAM,
        max_same_line_repeat=MAX_SAME_LINE_REPEAT,
    )

In [37]:
RANDOM_SEED = 425
TEMPERATURE = 0.3

START_PROMPT = "তারেক রহমান"

In [38]:
GENERATION_MODE = GENERATION_MODE.lower().strip()

if GENERATION_MODE == "both":
    print("\nGenerated poem (i. exact MLE):\n")
    print(generate_with_mode("mle"))
    print("\nGenerated poem (ii. seeded selection):\n")
    print(generate_with_mode("seeded"))
elif GENERATION_MODE in {"mle", "seeded"}:
    print(f"\nGenerated poem ({GENERATION_MODE}):\n")
    print(generate_with_mode(GENERATION_MODE))
else:
    raise ValueError('GENERATION_MODE must be "mle", "seeded", or "both"')


Generated poem (i. exact MLE):

তারেক রহমান। গুলে-লালা, নাচে তাকিয়া।
(ছায়ানট কাব্যগ্রন্থ)

Generated poem (ii. seeded selection):

তারেক রহমান। গুলে-বকৌলি- পরীক্ষা-কাহারবা
রংমহলের রংমশালাতে,
ডুবু ডুবু ডুবিল রে দেখ কত পারস্য-রাজার সনে
নিত্য-নূতনের গীতি,
একলা থাকার গানখানি সে গাবে -
উদাস পথিক ভাবে।
